# Week 5: Model Evaluation and Interpretation in Agricultural Analytics

**Objective:** Evaluate agricultural predictive models using appropriate metrics, cross-validation, error analysis, and interpretation techniques.

> **Dataset note:** The included dataset is a synthetic demonstration dataset created for this internship project. It is intended to demonstrate the evaluation workflow and should not be treated as field-validated agricultural data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
df = pd.read_csv("../data/agricultural_model_evaluation_dataset.csv")
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nSummary statistics:")
display(df.describe())


## 1. Regression Evaluation: Crop Yield Prediction

The regression target is `yield_tonnes_ha`. The model is evaluated using MAE, RMSE, and R².


In [ ]:
features = [
    "rainfall_mm", "temperature_c", "soil_ph", "nitrogen_kg_ha",
    "phosphorus_kg_ha", "potassium_kg_ha", "irrigation_mm",
    "pest_risk_index"
]
X = df[features]
y = df["yield_tonnes_ha"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

reg_model = RandomForestRegressor(
    n_estimators=250, random_state=RANDOM_STATE, min_samples_leaf=2
)
reg_model.fit(X_train, y_train)
y_pred = reg_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.3f} tonnes/ha")
print(f"RMSE : {rmse:.3f} tonnes/ha")
print(f"R²   : {r2:.3f}")


In [ ]:
# Regression error analysis
errors = y_test - y_pred
print("Mean residual:", errors.mean().round(3))
print("Largest absolute error:", np.abs(errors).max().round(3))

plt.figure(figsize=(7,5))
plt.scatter(y_test, y_pred, alpha=0.75)
plt.xlabel("Actual yield (tonnes/ha)")
plt.ylabel("Predicted yield (tonnes/ha)")
plt.title("Actual vs Predicted Crop Yield")
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(7,5))
plt.hist(errors, bins=20)
plt.xlabel("Residual (actual - predicted)")
plt.ylabel("Frequency")
plt.title("Crop Yield Residual Distribution")
plt.grid(alpha=0.25)
plt.show()


## 2. Cross-Validation

Five-fold cross-validation is used on the training dataset to estimate how consistently the regression model performs across different subsets.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = cross_validate(
    reg_model, X_train, y_train, cv=cv,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    }
)

print("Mean CV MAE :", -cv_results["test_MAE"].mean())
print("Mean CV RMSE:", -cv_results["test_RMSE"].mean())
print("Mean CV R²  :", cv_results["test_R2"].mean())


In [ ]:
# Feature importance
importance = pd.Series(
    reg_model.feature_importances_, index=features
).sort_values(ascending=False)

print(importance)

plt.figure(figsize=(8,5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.show()


## 3. Classification Evaluation: Disease Detection

The classification target is `disease_present`. Accuracy, precision, recall, F1-score and the confusion matrix are calculated.


In [ ]:
X_cls = df[features]
y_cls = df["disease_present"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cls
)

cls_model = RandomForestClassifier(
    n_estimators=250, random_state=RANDOM_STATE, class_weight="balanced"
)
cls_model.fit(Xc_train, yc_train)
yc_pred = cls_model.predict(Xc_test)

accuracy = accuracy_score(yc_test, yc_pred)
precision = precision_score(yc_test, yc_pred, zero_division=0)
recall = recall_score(yc_test, yc_pred, zero_division=0)
f1 = f1_score(yc_test, yc_pred, zero_division=0)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")

print("\nClassification report:")
print(classification_report(yc_test, yc_pred, zero_division=0))


In [ ]:
cm = confusion_matrix(yc_test, yc_pred)
print("Confusion Matrix:")
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No disease", "Disease"]
).plot()
plt.title("Disease Detection Confusion Matrix")
plt.show()


In [ ]:
# Stratified 5-fold cross-validation for classification
scv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cls_cv = cross_validate(
    cls_model, Xc_train, yc_train, cv=scv,
    scoring=["accuracy", "precision", "recall", "f1"]
)

print("Mean CV Accuracy :", cls_cv["test_accuracy"].mean())
print("Mean CV Precision:", cls_cv["test_precision"].mean())
print("Mean CV Recall   :", cls_cv["test_recall"].mean())
print("Mean CV F1       :", cls_cv["test_f1"].mean())


## 4. Interpretation and Agricultural Meaning

- **MAE** represents the typical absolute crop-yield prediction error in tonnes/ha.
- **RMSE** gives additional emphasis to large prediction errors.
- **R²** indicates how much variation in crop yield is explained by the model.
- **Precision** is important when false disease alerts could lead to unnecessary interventions.
- **Recall** is important when missing a real disease case could result in crop loss.
- **F1-score** balances precision and recall.
- The **confusion matrix** shows exactly which types of classification errors occurred.

High error may be caused by weather variability, measurement errors, missing management variables, unusual farms, limited training data, or distribution differences between training and deployment environments.


## 5. Limitations and Future Improvements

1. The dataset is synthetic and is not a substitute for field observations.
2. More seasons and geographic regions should be included for real deployment.
3. Farm-level or location-aware validation may be required when observations are correlated.
4. Time-aware validation should be considered for future forecasting tasks.
5. More management variables such as crop variety, sowing date, fertilizer application and irrigation schedule could improve the model.
6. Model performance should be monitored after deployment because agricultural conditions can change.


## 6. Final Conclusion

The evaluation workflow demonstrates how agricultural models can be assessed using multiple complementary metrics rather than a single score. Combining held-out testing, cross-validation, error analysis, confusion matrices and feature interpretation provides a stronger basis for judging model reliability. In a real agricultural project, the next step would be validation with independent field data and agricultural-domain expertise.
